In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, confusion_matrix,
                             matthews_corrcoef, cohen_kappa_score, log_loss,
                             mean_squared_error, mean_absolute_error, r2_score)
import time
import os
 
# Set the number of K folds as a global variable
K_FOLDS = 2
 
# Read the dataset from CSV file
df = pd.read_csv('../../../datasets/dataset-1.csv')
 
# Take 20% of the data
df = df.sample(frac=0.2, random_state=42)
 
# Rename the last column as 'label'
df.rename(columns={'GradeClass': 'label'}, inplace=True)
 
# Drop StudentID as it's just an identifier
X = df.drop(columns=['label', 'StudentID']).values
y = df['label'].values
 
timing_results = []
 
# Define classifiers and metrics
classifiers = {
    'DecisionTree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier(),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'GaussianProcess': GaussianProcessClassifier(kernel=RBF(1.0), random_state=42, max_iter_predict=100),
}

# Store results
results = []
 
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
 
# Create a directory to save confusion matrices
os.makedirs("confusion_matrices", exist_ok=True)
 
# Helper function for confusion matrix metrics
def confusion_matrix_metrics(cm, classes):
    metrics = {}
    for idx, class_label in enumerate(classes):
        TP = cm[idx, idx]  # True Positives for this class
        FP = cm[:, idx].sum() - TP  # False Positives for this class
        FN = cm[idx, :].sum() - TP  # False Negatives for this class
        TN = cm.sum() - (TP + FP + FN)  # True Negatives for this class
 
        metrics[class_label] = {
            'TPR': TP / (TP + FN + 1e-10) if (TP + FN) > 0 else 0,
            'TNR': TN / (TN + FP + 1e-10) if (TN + FP) > 0 else 0,
            'FPR': FP / (FP + TN + 1e-10) if (FP + TN) > 0 else 0,
            'FNR': FN / (FN + TP + 1e-10) if (FN + TP) > 0 else 0
        }
    return metrics
 
# Iterate over classifiers
for clf_name, clf in classifiers.items():
    fold_idx = 1
    for train_index, test_index in kf.split(X):
        # Split the data
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        print(f"Training {clf_name} on fold {fold_idx}...")
        
        # Record start time
        start_train_time = time.time()
        
        try:
            # For Gaussian Process, we might need to handle large datasets differently
            if clf_name == 'GaussianProcess' and len(X_train) > 1000:
                # Subsample for GP if dataset is too large
                subsample_indices = np.random.choice(len(X_train), size=1000, replace=False)
                clf.fit(X_train[subsample_indices], y_train[subsample_indices])
            else:
                clf.fit(X_train, y_train)
                
            train_time = time.time() - start_train_time
            
            start_test_time = time.time()
            y_pred = clf.predict(X_test)
            test_time = time.time() - start_test_time
            
            timing_results.append({
                'Classifier': clf_name,
                'Fold': fold_idx,
                'Training Time (s)': train_time,
                'Testing Time (s)': test_time,
                'Total Time (s)': train_time + test_time
            })
            
            # Compute metrics
            unique_classes = np.unique(np.concatenate((y_train, y_test)))
            cm = confusion_matrix(y_test, y_pred, labels=unique_classes)
            cm_metrics = confusion_matrix_metrics(cm, unique_classes)
            
            # Calculate overall metrics for this fold
            overall_metrics = {
                'Classifier': clf_name,
                'Fold': fold_idx,
                'Class': 'Overall',
                'Accuracy': accuracy_score(y_test, y_pred),
                'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
                'Recall': recall_score(y_test, y_pred, average='weighted'),
                'F1 Score': f1_score(y_test, y_pred, average='weighted'),
                'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
                'Matthews Correlation': matthews_corrcoef(y_test, y_pred),
                'Cohen Kappa': cohen_kappa_score(y_test, y_pred),
                'Training Time (s)': train_time,
                'Testing Time (s)': test_time
            }
            results.append(overall_metrics)
            
            class_metrics_list = []
            
            for class_label in unique_classes:
                bin_y_test = (y_test == class_label)
                bin_y_pred = (y_pred == class_label)
                
                class_specific_metrics = {
                    'Classifier': clf_name,
                    'Fold': fold_idx,
                    'Class': class_label,
                    'Accuracy': accuracy_score(bin_y_test, bin_y_pred),
                    'Precision': precision_score(bin_y_test, bin_y_pred, average='binary', zero_division=0),
                    'Recall': recall_score(bin_y_test, bin_y_pred, average='binary'),
                    'F1 Score': f1_score(bin_y_test, bin_y_pred, average='binary'),
                    'Balanced Accuracy': balanced_accuracy_score(bin_y_test, bin_y_pred),
                    'True Positive Rate (TPR)': cm_metrics[class_label]['TPR'],
                    'True Negative Rate (TNR)': cm_metrics[class_label]['TNR'],
                    'False Positive Rate (FPR)': cm_metrics[class_label]['FPR'],
                    'False Negative Rate (FNR)': cm_metrics[class_label]['FNR'],
                    'Training Time (s)': train_time,
                    'Testing Time (s)': test_time
                }
                
                class_metrics_list.append(class_specific_metrics)
            
            # Plot and save confusion matrix
            plt.figure(figsize=(8, 6))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=unique_classes, yticklabels=unique_classes)
            plt.title(f"{clf_name} - Fold {fold_idx} Confusion Matrix")
            plt.xlabel("Predicted")
            plt.ylabel("True")
            plt.savefig(f"confusion_matrices/{clf_name}_fold_{fold_idx}.png")
            plt.close()
            
            # Append results for this fold
            results.extend(class_metrics_list)
            
        except Exception as e:
            print(f"Error with {clf_name} on fold {fold_idx}: {str(e)}")
            # Add an entry with error information
            results.append({
                'Classifier': clf_name,
                'Fold': fold_idx,
                'Class': 'ERROR',
                'Error': str(e)
            })
            
        fold_idx += 1
 
timing_df = pd.DataFrame(timing_results)
timing_df.to_csv("time_with_gp.csv", index=False)
 
# Create a DataFrame for results
results_df = pd.DataFrame(results)
print("Classification Metrics Across Folds:")
print(results_df)
 
# Save results to CSV
results_df.to_csv("metrics_with_gp.csv", index=False)

# Create summary statistics - Fixed to avoid the groupby error
summary_metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Balanced Accuracy', 'Training Time (s)', 'Testing Time (s)']
summary_data = []

# Filter overall results
overall_results = results_df[results_df['Class'] == 'Overall']

# Calculate summary statistics for each classifier
for clf_name in classifiers.keys():
    clf_data = overall_results[overall_results['Classifier'] == clf_name]
    
    if not clf_data.empty:
        summary_row = {'Classifier': clf_name}
        
        for metric in summary_metrics:
            if metric in clf_data.columns:
                values = clf_data[metric].values
                if len(values) > 0:
                    summary_row[f'{metric}_mean'] = np.mean(values)
                    summary_row[f'{metric}_std'] = np.std(values)
                else:
                    summary_row[f'{metric}_mean'] = np.nan
                    summary_row[f'{metric}_std'] = np.nan
        
        summary_data.append(summary_row)

# Create summary DataFrame
summary_df = pd.DataFrame(summary_data)
print("\nSummary Statistics:")
print(summary_df)
summary_df.to_csv("summary_with_gp.csv", index=False)

# Plot performance comparison
plt.figure(figsize=(12, 6))
mean_metrics = {}
for clf_name in classifiers.keys():
    clf_data = overall_results[overall_results['Classifier'] == clf_name]
    if not clf_data.empty:
        mean_metrics[clf_name] = {
            metric: clf_data[metric].mean() for metric in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Balanced Accuracy']
            if metric in clf_data.columns
        }

# Create performance comparison dataframe
performance_df = pd.DataFrame(mean_metrics).T
if not performance_df.empty:
    performance_df.plot(kind='bar', figsize=(12, 6))
    plt.title('Model Performance Comparison')
    plt.ylabel('Score')
    plt.ylim(0, 1)
    plt.savefig('model_comparison.png')
    plt.close()

# Plot timing comparison
plt.figure(figsize=(10, 6))
timing_means = {}
for clf_name in classifiers.keys():
    clf_timing = timing_df[timing_df['Classifier'] == clf_name]
    if not clf_timing.empty:
        timing_means[clf_name] = {
            'Training Time (s)': clf_timing['Training Time (s)'].mean(),
            'Testing Time (s)': clf_timing['Testing Time (s)'].mean(),
            'Total Time (s)': clf_timing['Total Time (s)'].mean()
        }

# Create timing comparison dataframe
timing_comparison_df = pd.DataFrame(timing_means).T
if not timing_comparison_df.empty:
    timing_comparison_df.plot(kind='bar', figsize=(10, 6))
    plt.title('Model Timing Comparison')
    plt.ylabel('Time (seconds)')
    plt.savefig('timing_comparison.png')
    plt.close()

Training DecisionTree on fold 1...
Training DecisionTree on fold 2...
Training KNN on fold 1...
Training KNN on fold 2...
Training LogisticRegression on fold 1...


/home/yashbuddhadev21/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training LogisticRegression on fold 2...


/home/yashbuddhadev21/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training GaussianProcess on fold 1...
Training GaussianProcess on fold 2...
Classification Metrics Across Folds:
            Classifier  Fold    Class  Accuracy  Precision    Recall  \
0         DecisionTree     1  Overall  0.782427   0.801828  0.782427   
1         DecisionTree     1      0.0  0.941423   0.428571  0.500000   
2         DecisionTree     1      1.0  0.945607   0.823529  0.583333   
3         DecisionTree     1      2.0  0.866109   0.627451  0.711111   
4         DecisionTree     1      3.0  0.912134   0.709091  0.886364   
5         DecisionTree     1      4.0  0.899582   0.941176  0.842105   
6         DecisionTree     2  Overall  0.857741   0.858726  0.857741   
7         DecisionTree     2      0.0  0.966527   0.666667  0.400000   
8         DecisionTree     2      1.0  0.928870   0.633333  0.760000   
9         DecisionTree     2      2.0  0.958159   0.894737  0.850000   
10        DecisionTree     2      3.0  0.941423   0.868421  0.785714   
11        DecisionTree 

<Figure size 1200x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>